# 02.2 - Searching & Binary Search

## Linear search vs binary search
| | Linear search | Binary search |
|---|---|---|
| Requirement | none | **sorted** data (or a monotonic condition) |
| Time | O(n) | **O(log n)** |
| 1 billion items | 1,000,000,000 checks | **~30 checks** |

Binary search: look at the middle; if the target is bigger, discard the left half, otherwise discard the right half. Repeat.

## Where is it used?
| Real system | Use |
|---|---|
| **Database indexes** (B-trees are a multi-way binary search) | Find a row among billions |
| **`git bisect`** | Find the commit that introduced a bug in O(log n) builds |
| Dictionaries, phone books, sorted logs | Look up by key / timestamp |
| Python `bisect`, Java `Arrays.binarySearch` | Built into standard libraries |
| Capacity planning ("minimum servers to handle the load") | **Binary search on the answer** |
| Debugging by halving (comment out half the code) | Same idea! |

## 1. Linear search & three binary search templates

In [1]:
def linear_search(nums, target):          # O(n)
    for i, x in enumerate(nums):
        if x == target:
            return i
    return -1

def binary_search(nums, target):          # Template 1: find exact value
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

def lower_bound(nums, target):            # Template 2: first index with nums[i] >= target
    lo, hi = 0, len(nums)                 # note: hi = len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

def binary_search_recursive(nums, target, lo=0, hi=None):
    if hi is None:
        hi = len(nums) - 1
    if lo > hi:
        return -1
    mid = (lo + hi) // 2
    if nums[mid] == target:
        return mid
    if nums[mid] < target:
        return binary_search_recursive(nums, target, mid + 1, hi)
    return binary_search_recursive(nums, target, lo, mid - 1)

nums = [1, 3, 5, 5, 5, 8, 13]
print(linear_search(nums, 8), binary_search(nums, 8), binary_search_recursive(nums, 13), binary_search(nums, 4))
print("lower_bound(5) =", lower_bound(nums, 5), " lower_bound(6) =", lower_bound(nums, 6))

import bisect
assert lower_bound(nums, 5) == bisect.bisect_left(nums, 5)

5 5 6 -1
lower_bound(5) = 2  lower_bound(6) = 5


### Template 3 - Binary search on the answer When the question asks for the **minimum (or maximum) value that satisfies a condition**, and the condition is **monotonic** (once true, always true for bigger values):
```
F F F F T T T T T      <- find the first T
```
Search over the **range of possible answers**, not over the array.

In [2]:
def first_true(lo, hi, condition):
    # smallest x in [lo, hi] with condition(x) True (assumes condition(hi) is True)
    while lo < hi:
        mid = (lo + hi) // 2
        if condition(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

# Example: integer square root of 1000 -> largest x with x*x <= 1000
print(first_true(0, 1000, lambda x: x * x > 1000) - 1)   # 31

31


---
## Solved problems

### 1. Search Insert Position - LC #35 (Easy)
Index of target, or where it would be inserted.

**Idea:** This is exactly `lower_bound`.

In [3]:
def search_insert(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

assert search_insert([1, 3, 5, 6], 5) == 2
assert search_insert([1, 3, 5, 6], 2) == 1
assert search_insert([1, 3, 5, 6], 7) == 4
print("Search Insert ")

Search Insert 


### 2. Find First and Last Position - LC #34 (Medium) Return `[first, last]` index of target, or `[-1, -1]`, in O(log n).

**Idea:** Two lower-bound searches: first position of `target` and first position of `target + 1`.

In [4]:
def search_range(nums, target):
    def lower(t):
        lo, hi = 0, len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] < t:
                lo = mid + 1
            else:
                hi = mid
        return lo
    start = lower(target)
    if start == len(nums) or nums[start] != target:
        return [-1, -1]
    return [start, lower(target + 1) - 1]

assert search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert search_range([], 0) == [-1, -1]
print("First and Last ")

First and Last 


### 3. Sqrt(x) - LC #69 (Easy)
Integer square root without `**` or `math.sqrt`.

In [5]:
def my_sqrt(x):
    lo, hi = 0, x
    while lo <= hi:
        mid = (lo + hi) // 2
        if mid * mid <= x:
            lo = mid + 1
        else:
            hi = mid - 1
    return hi           # largest mid with mid*mid <= x

assert [my_sqrt(x) for x in [0, 1, 4, 8, 15, 16, 2147395599]] == [0, 1, 2, 2, 3, 4, 46339]
print("Sqrt ")

Sqrt 


### 4. Search a 2D Matrix - LC #74 (Medium)
Rows sorted, each row starts after the previous row ends. Search in O(log(m·n)).

**Idea:** Treat the matrix as one sorted array of length m·n; index `i` → `(i // cols, i % cols)`.

In [6]:
def search_matrix(matrix, target):
    rows, cols = len(matrix), len(matrix[0])
    lo, hi = 0, rows * cols - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        value = matrix[mid // cols][mid % cols]     # index -> (row, col)
        if value == target:
            return True
        if value < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

m = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert search_matrix(m, 3) and not search_matrix(m, 13)
print("Search 2D Matrix ")

Search 2D Matrix 


### 5. Find Minimum in Rotated Sorted Array - LC #153 (Medium)


**Idea:** Compare `nums[mid]` with `nums[hi]`: if bigger, the rotation point (minimum) is on the right.

In [7]:
def find_min(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1        # min is to the right of mid
        else:
            hi = mid            # mid could be the min
    return nums[lo]

assert find_min([3, 4, 5, 1, 2]) == 1
assert find_min([4, 5, 6, 7, 0, 1, 2]) == 0
assert find_min([11, 13, 15, 17]) == 11
print("Find Min Rotated ")

Find Min Rotated 


### 6. Capacity To Ship Packages Within D Days - LC #1011 (Medium) real logistics problem
Packages must ship in order. Minimum ship capacity to ship everything within `days` days.

**Idea:** Binary search on the answer. Capacity is between `max(weights)` and `sum(weights)`; if a capacity works, every bigger one works.

In [8]:
def ship_within_days(weights, days):
    def can_ship(capacity):
        needed, load = 1, 0
        for w in weights:
            if load + w > capacity:
                needed += 1
                load = 0
            load += w
        return needed <= days

    lo, hi = max(weights), sum(weights)      # answer range
    while lo < hi:
        mid = (lo + hi) // 2
        if can_ship(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

# Time O(n log(sum))
assert ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15
assert ship_within_days([3, 2, 2, 4, 1, 4], 3) == 6
assert ship_within_days([1, 2, 3, 1, 1], 4) == 3
print("Ship Within Days ")

Ship Within Days 


## More practice on LeetCode
Easy: First Bad Version (#278 - literally `git bisect`), Guess Number Higher or Lower (#374), Valid Perfect Square (#367)
Medium: Find Peak Element (#162), Time Based Key-Value Store (#981), Koko Eating Bananas (#875 - in 03.1)
Hard: Median of Two Sorted Arrays (#4), Split Array Largest Sum (#410)

## Key takeaways
- Sorted input or "minimum value that works" → think **binary search**.
- Learn the three templates: exact match, lower bound, search on the answer.
- Watch the loop condition (`<=` vs `<`) and the updates (`mid ± 1` vs `mid`) - off-by-one bugs live here.

**Next: `02.3-Sorting-Algorithms.ipynb`**